**Author(s):** Martin Ross, Oliver Baker  
**Affiliation(s):** Earth and Environmental Sciences, University of Waterloo  
**Date:** 2026-05-06 

---

## Overview
Code used for the Random Forest analysis in submitted manuscript to Quaternary Science Reviews titled 
"Improving regional paleo-ice sheet reconstructions with till provenance analysis: A new approach applied to northern Quebec and Labrador" 

## Data
Will be updated after manuscript acceptance

## Note on AI use
ChapGPT (GPT-5) was used to improve and clean the code, fix some issues, and for annotations

## Setup

In [ ]:
import numpy as np
import pandas as pd
import scipy
import scipy.stats as stats
from scipy.stats import gmean, probplot
from sklearn.cluster import KMeans, k_means
from sklearn import metrics
import matplotlib.pyplot as plt
import composition_stats as com
from composition_stats import closure, clr, centralize, center, multiplicative_replacement
#%matplotlib inline
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import MinMaxScaler, MaxAbsScaler, StandardScaler, scale, RobustScaler
from mpl_toolkits.mplot3d import Axes3D

 
# Functions:

# Data Transformation

def read_csv_file(file_path):
    """Read a .csv file and return a Pandas DataFrame."""
    
    return pd.read_csv(file_path)

def create_histogram_and_probabiliy_plots(array, data_type, fit):
    """Creates a histogram and probability plot for the array, using the data_type to title the plot and name the file.
    Line of best fit depends on fit = True/False."""
    
    # Create histogram
    n, bins, patches = plt.hist(array, bins='auto', density=True, alpha=0.9, edgecolor='black', linewidth=1.2)
    plt.xlabel('Values')
    plt.ylabel('Frequency')
    plt.title('Histogram of %s Data' % (data_type))
    # Save histogram
    if save == True:
        plt.savefig('%s_Hist_Data.png' % (name), format='png')
    plt.show()
    plt.close()
        
    # Create probability plot
    fig, ax = plt.subplots()
    stats.probplot(array, plot=ax, fit=fit)
    ax.set_title('Probability Plot of %s Data' % (data_type))
    # Save probability plot
    if save == True:
        plt.savefig('%s_Prob_%s_Data.png' % (name, data_type), format='png')
    plt.show()
    plt.close()

def remove_nondata(df):
    """Removes nondata columns from the df (dataframe). The ideal dataframe has headings which include: Sample, Latitude,
    Longitude, followed by selected elemental proportions for analysis (for example: Bi, Ag, As, Cu, Zn, etc.).
    Proportions should have consistent units (ppm, %)."""
    
    df = df[df.columns[~df.columns.isin(['Sample', 'Latitude', 'Longitude', 'Datum', 'Easting', 'Northing', 'Type'])]]

    df = df.dropna(axis=1)
    print(df)
    return df

def data_exploration(df):
    """Prepares a dataframe for transformation, and creates a histogram and probability plot of the flattened data."""
    
    # Create an array from dataset
    arr=df2.to_numpy()
    arr_closure=closure(arr)

    # Flatten the 2d array into a 1d array
    arr_closure_flat = arr_closure.flatten()
    
    # Create histogram and probability plot of flattened data
    create_histogram_and_probabiliy_plots(arr_closure_flat, 'Flattened', False)
    
    return arr_closure
    
def clr_transform(array):
    """Applies a CLR (centered log ratio) transformation and returns an array. This converts positive and dependent
    values to independent and ranging from negative to positive, to be used in multivariate analysis techniques."""
    
    # Apply a CLR transformation to the data
    arr_clr_transformed = clr(array)
    print("arr_clr_transformed Mean: %s \narr_clr_transformed Std: %s" % (arr_clr_transformed.mean(), arr_clr_transformed.std()))
    
    # Flatten the 2d array into a 1d array
    arr_clr_transformed_flat = arr_clr_transformed.flatten()
    
    # Create histogram and probability plot of CLR-transformed data
    create_histogram_and_probabiliy_plots(arr_clr_transformed_flat, 'CLR-Transformed', True)
    
    return arr_clr_transformed

def standardize_data(array):
    """ Standardizes and returns an array to a zero mean and unit variance to fit the requirements of PCA."""
    
    scaler = StandardScaler()
    arr_standardized = scaler.fit_transform(array)
    print("arr_standardized Mean: %s \narr_standardized Std: %s" % (arr_standardized.mean(), arr_standardized.std()))
    
    SD_data = pd.DataFrame(arr_standardized)
    if save == True:
        SD_data.to_csv('%s_SD_data.csv' % (name))
    
    arr_standardized_flat = arr_standardized.flatten()
    create_histogram_and_probabiliy_plots(arr_standardized_flat, 'Standardized', True)
    
    # returns both the standardized data and the scaler
    return arr_standardized, scaler


# Principal Component Analysis

def pca_initial():
    """Calculates and displays components' explained variance from the prepared array, and returns a dataframe."""
    
    # Calculating the 95% variance
    total_variance = sum(pca.explained_variance_)
    print('Total variance in the dataset is:', total_variance)
    var_95 = total_variance*0.95
    print('The 95% variance is: ', var_95, '\n')
    
    #Creating a df with the components and explained variance
    a = zip(range(0,n_components), pca.explained_variance_)
    a = pd.DataFrame(a, columns=['PCA Comp', 'Explained Variance'])
    
    #Trying to find 95%... 
    print('Variance explained with 2 components:', sum(a['Explained Variance'][0:2]))
    print('Variance explained with 3 components:', sum(a['Explained Variance'][0:3]))
    print('Variance explained with 4 components:', sum(a['Explained Variance'][0:4]))
    print('Variance explained with 5 components:', sum(a['Explained Variance'][0:5]))
    print('Variance explained with 6 components:', sum(a['Explained Variance'][0:6]), '\n')
    
    print(a, '\n')
        
    return a

def calculate_explained_variance_percent():
    """Calculates and returns a list of the percentages of variance for each PC (up to 5)."""
    
    PC_explained_list=[]
    print('Principal components and their explained variance (%):')
    for PC in range(0, 9):
        PC_Explained = (pca.explained_variance_ratio_[PC]*100).round(2)
        PC_explained_list.append(PC_Explained)
        print('PC%s: %s' % (PC+1, PC_Explained))
    return PC_explained_list

def create_PC_scree_plot():
    """Creates a scree plot from principal components and their explained ratios. This will help the user decide the
    number of principal components to use."""
    
    # Calculate the cumulative sum of explained variance ratio
    cumulative_var_ratio = np.cumsum(pca.explained_variance_ratio_)
    
    # Find the index where the cumulative sum first exceeds 0.95
    threshold_idx = np.where(cumulative_var_ratio >= 0.95)[0]
    
    if threshold_idx.size == 0:
        # If no index is found, set the threshold index to the last index
        threshold_idx = len(cumulative_var_ratio) - 1
    else:
        # Get the first index where the cumulative sum exceeds 0.95
        threshold_idx = threshold_idx[0]
        
    # Create a scree plot
    fig, ax1 = plt.subplots(1, figsize=(16, 6))
    x_axis = np.arange(len(a))
    ax1.plot(x_axis, pca.explained_variance_ratio_, linewidth=2, c='r')
    ax1.set_xticks(x_axis)
    ax1.set_xticklabels(x_axis+1)
    ax1.set_xlabel('n components')
    ax1.set_ylabel('explained ratio')
    
    # Add a horizontal dotted line at 95% explained variance ratio
    ax1.axhline(y=pca.explained_variance_ratio_[threshold_idx], linestyle=':',label='95% explained variance', c='blue')

    # Add legend and display the plot
    ax1.legend(prop=dict(size=20))

    if save == True:
        plt.savefig('%s_PCA-ScreePlot.png' % (name), format='png')
    plt.show()
    
def create_PC_columns():
    """Returns a list of PC column names (e.g., [PC1, PC2, PC3...])."""
    
    pc_columns=[]
    for n in range(0, n_components):
        pc_columns.append('PC%s' % (n+1))
    return pc_columns
    
def create_PC_loading_plot(loadings, pc_number):
    """Creates a PC loading plot, sorted by low to high values."""
    
    PC_sorted=loadings.sort_values(by=['PC%s' % (pc_number)])

    formatted_elements = [r"$%s$" % elem.replace('2', '_2').replace('3', '_3').replace('5', '_5') for elem in PC_sorted['Elements']]

    plt.figure(figsize=(12,8))                                  ## Changed to make trace elements fit on axis (12,8) before, now 15,6
    plt.bar(formatted_elements, PC_sorted['PC%s' % (pc_number)], linewidth=0.8, color='green')
    plt.ylim((-0.8, 0.8))
    plt.ylabel('Principal Component %s loadings' % (pc_number), fontsize=15)
    plt.axhline(0,linestyle='-', c='black')
    
    if save == True:
        plt.savefig('%s_PC%s_Loadings.png' % (name, pc_number), format='png')
    plt.show()
    plt.close()

def create_PC_loading_plots():
    """Creates muliple PC loading plots."""
    
    # Create loadings
    loadings = pd.DataFrame(pca.components_.T, columns=pc_columns, index=df2.columns)
    loadings.index.name='Elements' #To name the index column
    loadings['Elements']=loadings.index #the index is copied on to a new column with column name
    loadings = loadings.reset_index(drop=True) #the index replaced with sequence of numbers
    
    if save == True:
        loadings.to_csv('%s_Loading-Score.csv' % (name), index=False)
    
    # Creates all PC loading plots
    for n in range(1, n_components+1):
        create_PC_loading_plot(loadings, n)

# K-means Clustering

def create_Kmeans_scree_plot():
    """Creates a scree plot with inertia scores for each number of total clusters, from 2 to 20. The 'elbow' method
    is used to find the point of diminishing return. A sharp visual drop off, or elbow-shaped, on the plot 
    represents a much smaller difference in intertia scores, and therefore may be interpreted as a good number of 
    clusters to use. """
    
    no_of_clusters=range(2,20) #[2,3,4,5...]
    inertia=[]
    for f in no_of_clusters:
        kmeans = KMeans(n_clusters=f, random_state=42)
        kmeans = kmeans.fit(arr_X_r)
        u = kmeans.inertia_
        inertia.append(u)
        print('The inertia for', f, 'clusters is', u)
        
    fig, (ax1)=plt.subplots(1, figsize=(16,6))
    xx=np.arange(len(no_of_clusters))
    ax1.plot(xx, inertia)
    ax1.set_xticks(xx)
    ax1.set_xticklabels(no_of_clusters)
    plt.xlabel('Nb of clusters')
    plt.ylabel('Inertia score')
    # Plot a line for Nb. of clusters based on the 'elbow method'
    plt.axvline(2, color='red', linestyle=':')

    if save == True:
        plt.savefig('%s_Kmeans-ScreePlot.png' % (name), format='png')
        
def Calinski_Harabasz_scores():
    """Prints Calinski Harabasz scores for each number of clusters."""
    
    no_of_clusters=range(2,20) #[2,3,4,5...]
    Index2=[]    #Creates an empty list

    for i in no_of_clusters:
        kmeans=KMeans(n_clusters=i, random_state=42)
        kmeans=kmeans.fit(arr_X_r)
        CH=metrics.calinski_harabasz_score(arr_X_r, kmeans.labels_)
        Index2.append(CH)   # This will populate the inertia list with u
        print("The Calinski Harabasz score for :", i, "clusters is:", CH)
        ## highlight_samples=None


def create_PC_biplot(pc_x_axis, pc_y_axis, cluster_labels, cluster_columns, a_T,):
    """Creates PC biplots with data coloured by their K-means clusters."""
    
    y_num = np.array(cluster_labels)
    
    unique_cluster = np.unique(y_num)
    
    target_names = {i: f"Cluster {i}" for i in unique_cluster if i != -1}
    
    PC1_Explained = pca.explained_variance_ratio_[0].round(4) * 100
    PC2_Explained = pca.explained_variance_ratio_[1].round(4) * 100
    
    # Plotting the data
    plt.figure()
    plt.figure(figsize=(20,15))
    colors = ['magenta','black', 'blue', 'red', 'cyan', 'aqua', 'brown']
    lw=2
    pc_x_axis_col = pc_x_axis-1
    pc_y_axis_col = pc_y_axis-1
    # Get the absolute maximum value of PC scores
    PCx_abs_max = np.abs(arr_X_r[:,pc_x_axis_col]).max()
    PCy_abs_max = np.abs(arr_X_r[:,pc_y_axis_col]).max()
    n = pca.components_.T.shape[0]
    for color, i, target_name in zip(colors, range(1,number_of_clusters+1), target_names.values()):
        plt.scatter(arr_X_r[y_num == i, pc_x_axis_col], arr_X_r[y_num == i, pc_y_axis_col], color=color, alpha=.8,lw=lw, label=target_name)
        
    # if highlight_samples:
    #     for index, label in highlight_samples.items():
    #         x, y = arr_X_r[index, pc_x_axis_col], arr_X_r[index, pc_y_axis_col]
    #         plt.scatter(x, y, color='yellow', edgecolors='black', marker='*', s=200, zorder = 3)  # Make point visible
    #         plt.text(x, y, label, fontsize=12, color='black', ha='right', va='bottom', fontweight='bold')
        # plt.scatter(arr_X_r[highlight_samples, pc_x_axis_col], arr_X_r[highlight_samples, 
        #         pc_y_axis_col], color='yellow', edgecolors='black', marker='*', s=200, label="Highlighted Samples")
    
    
    ###Add the loading vectors scaled to the PC scores
    for i in range(n):
        plt.quiver(0, 0, pca.components_.T[i,pc_x_axis_col]*PCx_abs_max, pca.components_.T[i,pc_y_axis_col]*PCy_abs_max,color = 'b',alpha = 0.5, width=0.002, headwidth=4,headlength=4, angles='xy', scale_units='xy', scale=1)
  
    
    plt.legend(loc='best', shadow=False, scatterpoints=1, prop=dict(size=20))
    #plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.6)
    #This would used to put the legend outside the plot
    plt.xlabel(f'PC{pc_x_axis} [{PC_explained_list[pc_x_axis_col]}%]', fontsize=24)
    plt.ylabel(f'PC{pc_y_axis} [{PC_explained_list[pc_y_axis_col]}%]', fontsize=24  )
    plt.axvline(0,linestyle=':', c='black')
    plt.axhline(0, linestyle=':', c='black')
    for col in list(range(0, len(df2.columns))):
        ## Added *1.1 to move labels of arrows for better legibility
        plt.text(a_T[col,pc_x_axis_col]*PCx_abs_max*1.15, a_T[col,pc_y_axis_col]*PCy_abs_max, df2.columns[col], fontsize=15)
    plt.axvline(0,linestyle=':', c='black')
    plt.axhline(0, linestyle=':', c='black')
    
    if save == True:
        plt.savefig('%s_PC%s_PC%s_Biplot.pdf' % (name, pc_x_axis, pc_y_axis))
        plt.savefig('%s_PC%s_PC%s_Biplot.png' % (name, pc_x_axis, pc_y_axis), format='png')
    plt.show()
    plt.close()
        
    
def Kmeans_clustering():
    """Runs K-means clustering with the selected number of clusters, and produces PC1 v PC2 and PC1 v PC3 biplots.
    Also displays a table of the counts of each cluster, and saves a .csv file with each sample's PC scores and 
    associated K-means cluster."""

    # Apply KMeans with a fixed random state
    kmeans2=KMeans(n_clusters=number_of_clusters, random_state=42)  #because 42 is "the answer to the ultimate question of life, the universe, and everything."
    clusters2 = kmeans2.fit_predict(arr_X_r)
    
    # Predict the cluster labels for your data
    cluster_labels=kmeans2.fit_predict(arr_X_r)
    cluster_labels = cluster_labels +1
    
    # Create a DataFrame for cluster counts
    countsdbclusters = pd.DataFrame({'Cluster': pd.Series(clusters2).value_counts().index,
                                     'Count': pd.Series(clusters2).value_counts().values})
    
    print(countsdbclusters)
    
    # Optional: Print cluster centers for insight
    print("Cluster Centers:\n", kmeans2.cluster_centers_)

    # Calculating the counts of the clusters
    unique, counts=np.unique(kmeans2.labels_, return_counts=True)
    counts=counts.reshape(1,number_of_clusters) #one row and n columns (depending on number of clusters)

    # Creating a dataframe
    cluster_columns = []
    for n in range(0,number_of_clusters):
        cluster_columns.append('Cluster %s' % (n+1))
    countscldf2=pd.DataFrame(counts, columns=cluster_columns)

    # Display counts
    print('Number of data points in each cluster:')
    #display(countscldf2)
    
    a_=pca.components_.copy()
    a_T=a_.T
    
    # Print percentage of variance explained for each component
    print('Explained variance (first five components): %s' % str(pca.explained_variance_ratio_))
    
    # highlighted_samples=True 
    # highlighted_samples = { }
    
    # Create PC1 vs PC2 biplot : highlight_samples=highlighted_samples
    create_PC_biplot(1, 2, cluster_labels, cluster_columns, a_T)
    
    # Create PC1 vs PC3 biplot
    create_PC_biplot(1, 3, cluster_labels, cluster_columns, a_T)
    
    # 3D Plot
    # create_PC_biplot_3D(1, 2, 3, cluster_labels, cluster_columns, a_T)
    
    # Create a new dataframe of the PC scores
    df_PCscores=pd.DataFrame(arr_X_r, columns=pc_columns)
    
    # Insert the kmeans labels into the new dataframe
    df_PCscores.insert(len(pc_columns), "Cluster", cluster_labels)
    
    # Re-insert nondata
    df_PCscores.insert(0, "Easting", df["Easting"])
    df_PCscores.insert(0, "Northing", df["Northing"])
    df_PCscores.insert(0, "Sample", df["Sample"])
    
    if save == True:
        df_PCscores.to_csv('%s_PC-Score.csv' % (name), index=False)


In [ ]:
# User inputs: 
#  save, name, file path [3]
#  number_of_components [6]
#  number_of_clusters [9]

In [ ]:
# USER INPUTS

save = True  # Save figures and .csv files?
name = 'Name'  # This will be the prefix for produced figures and files
filepath = 'Till_Samples.csv'

## Data Transformation

In [ ]:
# Bring in the dataset
df = read_csv_file(filepath)

# Prepare the data for multivariate analysis
df2 = remove_nondata(df)
element_columns = list(df2.columns)
print(df2.columns.tolist())

arr_closure = data_exploration(df2)
arr_clr_transformed = clr_transform(arr_closure)

# Export CLR-transformed (non-standardized) dat
CLR_data = pd.DataFrame(arr_clr_transformed, columns=df2.columns)
CLR_data.to_csv(f'{name}_CLR_data.csv', index=False)
print(f"CLR-transformed data saved as: {name}_CLR_data.csv")

arr_standardized, scaler = standardize_data(arr_clr_transformed)

# Export standardized and CLR-transformed data
# STD_data = pd.DataFrame(arr_standardized, columns=df2.columns)
# STD_data.to_csv(f'{name}_STD_data.csv', index=False)
# print(f"Standardized data saved as: {name}_STD_data.csv")

# Change depending on need
# arr_X = arr_standardized
arr_X = arr_clr_transformed


## Principal Component Analysis

In [ ]:
n_components = arr_clr_transformed.shape[1]

# Running PCA on all components
pca=PCA(n_components=n_components)
arr_X_r=pca.fit_transform(arr_X)
a = pca_initial()

PC_explained_list = calculate_explained_variance_percent()

Calinski_Harabasz_scores()
create_PC_scree_plot()


In [ ]:
# Running PCA again, this time with the specified amount of components

# USER INPUT
n_components = 9

pca=PCA(n_components=n_components)
arr_X_r=pca.fit(arr_X).transform(arr_X)

# Generate PC loading plots
pc_columns = create_PC_columns()
create_PC_loading_plots()

## K-means Clustering

In [ ]:
# Deciding the number of clusters to use

# Making a scree plot of inertia scores; the elbow method to determine the number of clusters. The 'elbow' method
#   is used to find the point of diminishing return. The elbow point (\_-shaped) on the plot represents a much smaller 
#   difference in intertia scores, and therefore may be interpreted as a good number of clusters to use.

create_Kmeans_scree_plot()

In [ ]:
# Another method to determine the number of clusters (the Calinski-Harabasz score). In this method, the number of
#   clusters with the highest CH score is recommended.

Calinski_Harabasz_scores()

In [ ]:
# Create a k-means model with the specified amount of clusters 

# USER INPUT
number_of_clusters = 5

Kmeans_clustering()

## Fit Bedrock data to Till PCA Model

In [ ]:
# USER INPUTS
bedrock_filepath = 'Bedrock_Samples.csv'
bedrock_name = 'Name'

In [ ]:
# Data Transformation


# Read bedrock subset
bedrock = read_csv_file(bedrock_filepath)

print("Original bedrock subset shape:", bedrock.shape)
print("Original bedrock subset columns:")
print(bedrock.columns.tolist())

# Remove non-data columns
bedrock_df = bedrock[bedrock.columns[~bedrock.columns.isin(['Sample', 'Easting', 'Northing', 'Type'])]]
bedrock_df = bedrock_df.dropna(axis=1)
metadata_columns = []

# Select the same element columns used in the PCA model
missing_columns = []

for col in element_columns:
    if col not in bedrock.columns:
        missing_columns.append(col)

if len(missing_columns) > 0:
    raise ValueError("The bedrock file is missing these PCA element columns: %s" % missing_columns)

bedrock_elements = bedrock[element_columns].copy()

print("Selected bedrock element shape:", bedrock_elements.shape)
print("Number of PCA element columns:", len(element_columns))


# Convert selected element columns to numeric
bedrock_numeric = bedrock_elements.apply(pd.to_numeric, errors='coerce')

print("NaNs per selected element column:")
print(bedrock_numeric.isna().sum())

# Remove NaN
valid_mask = bedrock_numeric.notna().all(axis=1) & (bedrock_numeric > 0).all(axis=1)

print("Valid rows kept:", valid_mask.sum())
print("Invalid rows removed:", len(valid_mask) - valid_mask.sum())

bedrock_numeric_clean = bedrock_numeric[valid_mask].reset_index(drop=True)

if bedrock_numeric_clean.shape[0] == 0:
    raise ValueError("No valid bedrock rows remain after removing NaN, zero, or negative values.")

# Closure and CLR transformation
bedrock_array = bedrock_numeric_clean.to_numpy()

bedrock_closure = closure(bedrock_array)
bedrock_clr = clr(bedrock_closure)

# Keeps shape as 2D even if only one sample is projected
bedrock_clr = np.atleast_2d(bedrock_clr)

print("Bedrock CLR shape:", bedrock_clr.shape)

# Standardize using the SAME scaler fitted on the Till PCA data
bedrock_std = scaler.transform(bedrock_clr)

print("Bedrock standardized shape:", bedrock_std.shape)

In [ ]:
# Fit Bedrock data to Till PCA


# Change depending on need
bedrock_pca_input = bedrock_clr
# bedrock_pca_input = bedrock_std

# Project bedrock subset into the existing PCA model
bedrock_pca_scores = pca.transform(bedrock_pca_input)
bedrock_pc_columns = []

for n in range(0, bedrock_pca_scores.shape[1]):
    bedrock_pc_columns.append('PC%s' % (n + 1))

bedrock_PCscores = pd.DataFrame(bedrock_pca_scores, columns=bedrock_pc_columns)

# Re-insert nondata
bedrock_PCscores.insert(0, "Easting", bedrock["Easting"])
bedrock_PCscores.insert(0, "Northing", bedrock["Northing"])
bedrock_PCscores.insert(0, "Sample", bedrock["Sample"])
    

In [ ]:
# Save outputs
if save:
    # Save CLR Data
    bedrock_CLR_df = pd.DataFrame(bedrock_clr, columns=element_columns)
    bedrock_CLR_df.to_csv(f'{bedrock_name}_CLR.csv', index=False)
    
    # Save STD Data
    bedrock_STD_df = pd.DataFrame(bedrock_std, columns=element_columns)
    bedrock_STD_df.to_csv(f'{bedrock_name}_STD.csv', index=False)
    
    bedrock_PCscores.to_csv(f'{bedrock_name}_PC-Score.csv', index=False)